## Generate 2 samples per method/model given a task

In [1]:
print("Setting up environment...")
import torch
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
import sys
import os
import importlib

# Add src to path to import modules
sys.path.append('/storage/home/hcoda1/1/agupta886/scratch/diffusion-continual-learning')

# Import and reload the DDIM module to get the updated version with sample_with_noise
from src import ddim
from analysis.common import set_seed

# Reload the ddim module to get the latest changes
importlib.reload(ddim)
from src.ddim import build_conditional_ddim

print("DDIM import done and reloaded.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Setting up environment...


/storage/home/hcoda1/1/agupta886/scratch/python-envs/iclr-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DDIM import done and reloaded.
Using device: cuda


In [2]:
def get_dataset_config(dataset_name: str) -> Dict:
    """Get dataset configuration similar to generate_samples.py"""
    name = dataset_name.lower()
    if name == "mnist":
        return {"num_classes": 10, "channels": 1, "im_size": 32}
    elif name == "fmnist":
        return {"num_classes": 10, "channels": 1, "im_size": 32}
    elif name == "cifar10":
        return {"num_classes": 10, "channels": 3, "im_size": 32}
    elif name == "imagenet64": 
        return {"num_classes": 1000, "channels": 3, "im_size": 32}
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

def load_model(checkpoint_path: str, dataset_config: Dict) -> torch.nn.Module:
    """Load a diffusion model from checkpoint"""
    model = build_conditional_ddim(
        in_channel=dataset_config["channels"],
        image_size=dataset_config["im_size"],
        num_class_labels=dataset_config["num_classes"]
    ).to(device)
    
    ckpt_path = Path(checkpoint_path)
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    
    state = torch.load(ckpt_path, map_location=device)
    missing, unexpected = model.load_state_dict(state, strict=False)
    
    if missing:
        raise ValueError(f"Missing keys in checkpoint load: {missing}")
    if unexpected:
        print(f"[Warning] Unexpected keys: {unexpected}")
    
    model.eval()
    return model

In [3]:
def generate_fixed_noise(num_samples: int, 
                        channels: int, 
                        image_size: int, 
                        seed: int = 123,
                        device: torch.device = None) -> torch.Tensor:
    """Generate fixed noise tensors that will be reused across all methods/models"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    set_seed(seed)
    noise = torch.randn(num_samples, channels, image_size, image_size, device=device)
    return noise

def generate_samples_with_fixed_noise(model: torch.nn.Module, 
                                    task_class: int, 
                                    fixed_noise: torch.Tensor,
                                    num_inference_steps: int = 50,
                                    guidance_scale: float = 0.0) -> List[Image.Image]:
    """Generate samples using pre-generated fixed noise for consistent comparison"""
    num_samples = fixed_noise.shape[0]
    
    with torch.no_grad():
        # Use the model's sample_with_noise method with fixed noise
        batch = model.sample_with_noise(
            noise=fixed_noise,
            labels=[task_class] * num_samples,
            num_inference_steps=num_inference_steps,
            device=fixed_noise.device,
            guidance_scale=guidance_scale,
        )  # (B,C,H,W) in [-1,1]
    
    # Convert to PIL images
    pil_images = []
    for i in range(num_samples):
        t_im = batch[i].detach().cpu().clamp(-1, 1)
        pil = TF.to_pil_image((t_im + 1.0) * 0.5)
        pil_images.append(pil)
    
    return pil_images

def generate_samples(model: torch.nn.Module, 
                    task_class: int, 
                    num_samples: int = 2,
                    num_inference_steps: int = 50,
                    guidance_scale: float = 0.0,
                    seed: int = 123) -> List[Image.Image]:
    """Generate samples for a specific task/class using the loaded model (original function)"""
    set_seed(seed)
    
    with torch.no_grad():
        batch = model.sample(
            batch_size=num_samples,
            labels=[task_class] * num_samples,
            num_inference_steps=num_inference_steps,
            device=device,
            guidance_scale=guidance_scale,
        )  # (B,C,H,W) in [-1,1]
    
    # Convert to PIL images
    pil_images = []
    for i in range(num_samples):
        t_im = batch[i].detach().cpu().clamp(-1, 1)
        pil = TF.to_pil_image((t_im + 1.0) * 0.5)
        pil_images.append(pil)
    
    return pil_images

def find_model_checkpoints(method_dir: str, num_models: int = 20, min_model: int = 0) -> List[str]:
    """Find model checkpoints in a method directory starting from min_model"""
    method_path = Path(method_dir)
    if not method_path.exists():
        raise FileNotFoundError(f"Method directory not found: {method_dir}")
    
    # Look for .pt or .pth files starting from min_model
    checkpoints = []
    for i in range(min_model, min_model + num_models):
        # Try .pt first, then .pth
        pt_file = method_path / f"model-task{i}.pt"
        
        if pt_file.exists():
            checkpoints.append(pt_file)
        else:
            # If we can't find this model, stop looking for more
            break
    
    checkpoints = [str(cp) for cp in checkpoints]
    
    if len(checkpoints) < num_models:
        print(f"Warning: Found only {len(checkpoints)} checkpoints in {method_dir} starting from model {min_model}, expected {num_models}")
    return checkpoints[:num_models]

In [4]:
def create_method_visualization(method_samples: List[List[Image.Image]], 
                              method_name: str,
                              gap_width: int = 2,
                              samples_per_model: int = 2) -> Image.Image:
    """
    Create visualization for one method with samples arranged as:
    - N samples vertically per model (configurable)
    - models horizontally
    
    Args:
        method_samples: List of lists, each containing N PIL images
        method_name: Name of the method for labeling
        gap_width: Width of gap between models
        samples_per_model: Number of samples to stack vertically per model
    """
    if not method_samples:
        raise ValueError("No samples provided")
    
    # Get dimensions from first sample
    sample_height = method_samples[0][0].height
    sample_width = method_samples[0][0].width
    
    # Calculate total dimensions
    num_models = len(method_samples)
    total_width = num_models * sample_width + (num_models - 1) * gap_width
    total_height = samples_per_model * sample_height  # N samples stacked vertically
    
    # Create combined image
    combined = Image.new('RGB', (total_width, total_height), (255, 255, 255))
    
    x_offset = 0
    for model_idx, model_samples in enumerate(method_samples):
        if len(model_samples) != samples_per_model:
            print(f"Warning: Expected {samples_per_model} samples per model, got {len(model_samples)} for model {model_idx}")
            # Pad with dummy images if not enough samples
            while len(model_samples) < samples_per_model:
                dummy_img = Image.new('RGB', (sample_width, sample_height), (240, 240, 240))
                model_samples.append(dummy_img)
        
        # Place samples vertically
        for sample_idx in range(samples_per_model):
            if sample_idx < len(model_samples):
                y_pos = sample_idx * sample_height
                combined.paste(model_samples[sample_idx], (x_offset, y_pos))
        
        # Move to next model position - only add gap if not the last model
        x_offset += sample_width
        if model_idx < num_models - 1:  # Only add gap between models, not after the last one
            x_offset += gap_width
    
    return combined

def create_vertical_visualization(all_method_samples: Dict[str, List[List[Image.Image]]],
                                task_class: int,
                                method_gap: int = 10,
                                model_gap: int = 2,
                                samples_per_model: int = 2) -> Image.Image:
    """
    Create vertical visualization with methods stacked below each other
    
    Args:
        all_method_samples: Dict mapping method names to their samples
        task_class: The task/class number being visualized
        method_gap: Gap height between methods
        model_gap: Gap width between models within a method
        samples_per_model: Number of samples to stack vertically per model
    """
    method_gap = 5
    if not all_method_samples:
        raise ValueError("No method samples provided")
    
    # Create individual method visualizations
    method_images = {}
    max_width = 0
    total_height = 0
    
    for method_name, method_samples in all_method_samples.items():
        method_img = create_method_visualization(method_samples, method_name, model_gap, samples_per_model)
        method_images[method_name] = method_img
        max_width = max(max_width, method_img.width)
        total_height += method_img.height
    
    # Add gaps between methods
    total_height += (len(method_images) - 1) * method_gap
    
    # Create final combined image
    final_img = Image.new('RGB', (max_width, total_height), (255, 255, 255))
    
    # Stack method images vertically
    y_offset = 0
    for method_name, method_samples in all_method_samples.items():
        method_img = method_images[method_name]
        final_img.paste(method_img, (0, y_offset))
        y_offset += method_img.height + method_gap
    
    return final_img

In [5]:
def generate_qualitative_comparison(method_directories: Dict[str, str],
                                   dataset_name: str,
                                   task_class: int,
                                   num_models: int = 20,
                                   min_model: int = 0,
                                   num_samples_per_model: int = 2,
                                   num_inference_steps: int = 50,
                                   guidance_scale: float = 0.0,
                                   seed: int = 123,
                                   output_path: str = None,
                                   save_pdf: bool = True,
                                   use_fixed_noise: bool = True,
                                   model_gap: int = 2,
                                   method_gap: int = 10) -> Image.Image:
    """
    Main function to generate vertical qualitative comparison across methods
    
    Args:
        method_directories: Dict mapping method names to their checkpoint directories
        dataset_name: Name of dataset (mnist, cifar10, etc.)
        task_class: Class/task number to generate samples for
        num_models: Number of models per method (default 20)
        min_model: Starting model index (default 0)
        num_samples_per_model: Number of samples per model (2, 3, 4, etc.)
        num_inference_steps: DDIM inference steps
        guidance_scale: Guidance scale for generation
        seed: Random seed
        output_path: Optional path to save the final image
        save_pdf: Whether to save PDF version
        use_fixed_noise: Whether to use the same noise across all methods/models
        model_gap: Gap between models horizontally (within same method)
        method_gap: Gap between methods vertically
    
    Returns:
        PIL Image with the full comparison
    """
    print(f"Generating qualitative comparison for task {task_class} on {dataset_name}")
    print(f"Methods: {list(method_directories.keys())}")
    print(f"Models: {min_model} to {min_model + num_models - 1}")
    print(f"Samples per model: {num_samples_per_model}")
    print(f"Using fixed noise: {use_fixed_noise}")
    print(f"Model gap: {model_gap}px, Method gap: {method_gap}px")
    
    # Get dataset configuration
    dataset_config = get_dataset_config(dataset_name)
    
    # Generate fixed noise for consistent comparison if requested
    fixed_noise = None
    if use_fixed_noise:
        print(f"Generating fixed noise with seed {seed}...")
        fixed_noise = generate_fixed_noise(
            num_samples=num_samples_per_model,
            channels=dataset_config["channels"],
            image_size=dataset_config["im_size"],
            seed=seed,
            device=device
        )
        print(f"Fixed noise shape: {fixed_noise.shape}")
    
    # Store all method samples
    all_method_samples = {}
    
    for method_name, method_dir in method_directories.items():
        print(f"\nProcessing method: {method_name}")
        
        # Find checkpoints starting from min_model
        checkpoints = find_model_checkpoints(method_dir, num_models, min_model)
        print(f"Found {len(checkpoints)} checkpoints starting from model {min_model}")
        
        method_samples = []
        
        for i, checkpoint_path in enumerate(checkpoints):
            model_idx = min_model + i
            print(f"  Processing model {i+1}/{len(checkpoints)}: {Path(checkpoint_path).name} (task {model_idx})")
            
            try:
                # Load model
                model = load_model(checkpoint_path, dataset_config)
                
                # Generate samples
                if use_fixed_noise and fixed_noise is not None:
                    # Use fixed noise for consistent comparison
                    samples = generate_samples_with_fixed_noise(
                        model, 
                        task_class, 
                        fixed_noise,
                        num_inference_steps,
                        guidance_scale
                    )
                else:
                    # Use random noise (original behavior)
                    samples = generate_samples(
                        model, 
                        task_class, 
                        num_samples_per_model,
                        num_inference_steps,
                        guidance_scale,
                        seed  # Vary seed slightly for each model
                    )
                
                method_samples.append(samples)
                
                # Clean up GPU memory
                del model
                torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"    Error processing {checkpoint_path}: {e}")
                # Create dummy images if model fails to load
                dummy_img = Image.new('RGB', (dataset_config["im_size"], dataset_config["im_size"]), (128, 128, 128))
                dummy_samples = [dummy_img] * num_samples_per_model
                method_samples.append(dummy_samples)
        
        all_method_samples[method_name] = method_samples
    
    # Create vertical visualization with custom gaps
    print(f"\nCreating visualization...")
    final_image = create_vertical_visualization(all_method_samples, task_class, 
                                              method_gap=method_gap,
                                              model_gap=model_gap,
                                              samples_per_model=num_samples_per_model)
    
    # Save as PNG
    if output_path:
        final_image.save(output_path)
        print(f"Saved visualization to: {output_path}")
        
        # Save as PDF if requested
        if save_pdf:
            pdf_path = output_path.replace('.png', '.pdf')
            final_image.save(pdf_path, "PDF", resolution=300.0)
            print(f"Saved PDF to: {pdf_path}")
    
    return final_image

## Usage Example

Configure your method directories and generate comparisons:

## Helper Functions for Interactive Setup

If you need help finding your checkpoint directories or want to explore the structure:

In [6]:
def explore_directory(base_path: str, max_depth: int = 3) -> None:
    """Explore directory structure to help find checkpoint paths"""
    base = Path(base_path)
    if not base.exists():
        print(f"Directory does not exist: {base_path}")
        return
    
    print(f"Exploring {base_path}:")
    for root, dirs, files in os.walk(base):
        level = root.replace(str(base), '').count(os.sep)
        if level >= max_depth:
            dirs[:] = []  # Don't go deeper
            continue
        
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        
        # Show .pt and .pth files
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            if file.endswith(('.pt', '.pth')):
                print(f"{subindent}{file}")
        
        if level == 0 and len(files) > 10:  # Don't show too many files at root
            print(f"{subindent}... and {len(files)-10} more files")

def count_checkpoints_in_methods(method_dirs: Dict[str, str], min_model: int = 0) -> None:
    """Count available checkpoints in each method directory starting from min_model"""
    print(f"Checkpoint counts per method (starting from model {min_model}):")
    for method_name, method_dir in method_dirs.items():
        try:
            checkpoints = find_model_checkpoints(method_dir, num_models=20, min_model=min_model)
            print(f"  {method_name}: {len(checkpoints)} checkpoints")
        except Exception as e:
            print(f"  {method_name}: Error - {e}")

# Example usage:
# explore_directory("/your/base/path/to/experiments")
# count_checkpoints_in_methods(imagenet64_methods, min_model=1)

## ImageNet64 Method Comparison Setup

Configuration for the three ImageNet64 methods: standard, diagonal, and rank1_opt variants.

In [7]:
# ImageNet64 method directories - update the base path as needed
base_experiment_path = "/storage/home/hcoda1/1/agupta886/scratch/imagenet"  # Update this path

imagenet64_methods = {
    "GR-Distil": f"{base_experiment_path}/imagenet64--gr-distil",
    "Diag-GR-Distil": f"{base_experiment_path}/imagenet64-diag-gr-distil", 
    "Rank1-Opt-GR-Distil": f"{base_experiment_path}/imagenet64-rank1_opt-gr-distil"
}

# imagenet64_methods = {
#     "GR-Distil": f"{base_experiment_path}/imagenet64--gr-distil-345",
#     "Diag-GR-Distil": f"{base_experiment_path}/imagenet64-diag-gr-distil-345", 
#     "Rank1-Opt-GR-Distil": f"{base_experiment_path}/imagenet64-rank1_opt-gr-distil-345"
# }


print("ImageNet64 method configuration:")
for method_name, path in imagenet64_methods.items():
    print(f"  {method_name}: {path}")


ImageNet64 method configuration:
  GR-Distil: /storage/home/hcoda1/1/agupta886/scratch/imagenet/imagenet64--gr-distil
  Diag-GR-Distil: /storage/home/hcoda1/1/agupta886/scratch/imagenet/imagenet64-diag-gr-distil
  Rank1-Opt-GR-Distil: /storage/home/hcoda1/1/agupta886/scratch/imagenet/imagenet64-rank1_opt-gr-distil


In [8]:
# Check if directories exist and count checkpoints
min_model_check = 0  # Set this to check from a specific starting model
print("Checking ImageNet64 method directories and checkpoint counts:")
print("=" * 60)

for method_name, method_dir in imagenet64_methods.items():
    print(f"\n{method_name}:")
    print(f"  Path: {method_dir}")
    
    if Path(method_dir).exists():
        try:
            checkpoints = find_model_checkpoints(method_dir, num_models=20, min_model=min_model_check)
            print(f"  Status: ✓ Directory exists")
            print(f"  Checkpoints found: {len(checkpoints)} (starting from model {min_model_check})")
            if checkpoints:
                print(f"  First checkpoint: {Path(checkpoints[0]).name}")
                if len(checkpoints) > 1:
                    print(f"  Last checkpoint: {Path(checkpoints[-1]).name}")
        except Exception as e:
            print(f"  Status: ✗ Error reading directory: {e}")
    else:
        print(f"  Status: ✗ Directory does not exist")
        
print(f"\n{'='*60}")
print("Ready to generate comparisons once directories are confirmed!")

Checking ImageNet64 method directories and checkpoint counts:

GR-Distil:
  Path: /storage/home/hcoda1/1/agupta886/scratch/imagenet/imagenet64--gr-distil
  Status: ✓ Directory exists
  Checkpoints found: 20 (starting from model 0)
  First checkpoint: model-task0.pt
  Last checkpoint: model-task19.pt

Diag-GR-Distil:
  Path: /storage/home/hcoda1/1/agupta886/scratch/imagenet/imagenet64-diag-gr-distil
  Status: ✓ Directory exists
  Checkpoints found: 20 (starting from model 0)
  First checkpoint: model-task0.pt
  Last checkpoint: model-task19.pt

Rank1-Opt-GR-Distil:
  Path: /storage/home/hcoda1/1/agupta886/scratch/imagenet/imagenet64-rank1_opt-gr-distil
  Status: ✓ Directory exists
  Checkpoints found: 20 (starting from model 0)
  First checkpoint: model-task0.pt
  Last checkpoint: model-task19.pt

Ready to generate comparisons once directories are confirmed!


## Generate Comparison

Generate the vertical comparison with methods stacked below each other:

In [9]:
num_samples = 1  # Number of samples per model (vertically stacked)
min_model = 10
test_num_models = 10  # Start with fewer models for testing

print(f"Layout: Methods stacked vertically with {num_samples} samples per model")
print(f"Using {test_num_models} models per method for testing...")
print("🔒 USING FIXED_NOISE - Same initial noise for all methods/models!")
print("This may take several minutes...")
mkdir_name = f"final_images_{min_model}_more"
os.makedirs(mkdir_name, exist_ok=True)

for seed in range(10):  # Different seeds for different runs
    for target_class in [511, 531]:
        # Generate the comparison with FIXED NOISE
        final_image = generate_qualitative_comparison(
        method_directories=imagenet64_methods,
        dataset_name="imagenet64",
        task_class=target_class,
        num_models=test_num_models,
        min_model=min_model,  # Use fewer models for testing
        num_samples_per_model=num_samples,  # Vertically stacked samples
        output_path=f"final_images_{min_model}_more/imagenet64_class_{target_class:03d}_{num_samples}samples_{test_num_models}models_{seed}.png",
        save_pdf=True,
        use_fixed_noise=False,  # 🔒 This ensures same noise across all methods!
        seed=seed
        )

        print(f"Comparison complete!")
        print(f"🔒 Used FIXED NOISE - same initial noise transformed by each method!")
        print(f"PNG saved to: final_images_{min_model}_more/imagenet64_class_{target_class:03d}_{num_samples}samples_{test_num_models}models_{seed}.png")

Layout: Methods stacked vertically with 1 samples per model
Using 10 models per method for testing...
🔒 USING FIXED_NOISE - Same initial noise for all methods/models!
This may take several minutes...
Generating qualitative comparison for task 511 on imagenet64
Methods: ['GR-Distil', 'Diag-GR-Distil', 'Rank1-Opt-GR-Distil']
Models: 10 to 19
Samples per model: 1
Using fixed noise: False
Model gap: 2px, Method gap: 10px

Processing method: GR-Distil
Found 10 checkpoints starting from model 10
  Processing model 1/10: model-task10.pt (task 10)


  Processing model 2/10: model-task11.pt (task 11)
  Processing model 3/10: model-task12.pt (task 12)
  Processing model 4/10: model-task13.pt (task 13)
  Processing model 5/10: model-task14.pt (task 14)
  Processing model 6/10: model-task15.pt (task 15)
  Processing model 7/10: model-task16.pt (task 16)
  Processing model 8/10: model-task17.pt (task 17)
  Processing model 9/10: model-task18.pt (task 18)
  Processing model 10/10: model-task19.pt (task 19)

Processing method: Diag-GR-Distil
Found 10 checkpoints starting from model 10
  Processing model 1/10: model-task10.pt (task 10)
  Processing model 2/10: model-task11.pt (task 11)
  Processing model 3/10: model-task12.pt (task 12)
  Processing model 4/10: model-task13.pt (task 13)
  Processing model 5/10: model-task14.pt (task 14)
  Processing model 6/10: model-task15.pt (task 15)
  Processing model 7/10: model-task16.pt (task 16)
  Processing model 8/10: model-task17.pt (task 17)
  Processing model 9/10: model-task18.pt (task 18)
  